In [15]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)

True

In [2]:
params = {"command": "npx", "args": ["-y", "mcp-memory-libsql"], "env": {"LIBSQL_URL": "file:./memory/ed.db"}}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='create_entities', description='Create new entities with observations', inputSchema={'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string'}, 'entityType': {'type': 'string'}, 'observations': {'type': 'array', 'items': {'type': 'string'}}}, 'required': ['name', 'entityType', 'observations']}}}, 'required': ['entities'], '$schema': 'http://json-schema.org/draft-07/schema#'}, annotations=None, title='Create new entities with observations'),
 Tool(name='search_nodes', description='Search for entities and their relations using text search with relevance ranking', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string'}, 'limit': {'type': 'number'}}, 'required': ['query'], '$schema': 'http://json-schema.org/draft-07/schema#'}, annotations=None, title='Search for entities and their relations using text search with relevance ranking'),
 Tool(name='read_graph', description='Get recent entit

In [10]:
instructions = """You use your entity tools as a persistent memory to store and recall information about your conversations.
IMPORTANT: Before answering ANY question about a person or topic, you MUST first use search_nodes or read_graph to check your memory. Never say you don't know something without checking memory first.
"""
request = "My name's Ed. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. \
MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = "gpt-4.1-mini"

In [11]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", model=model, instructions=instructions, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Nice to meet you, Ed! It sounds like you have a fascinating course on AI Agents and the MCP protocol. If there's anything specific you'd like to discuss or any questions you have about AI Agents, MCP, or related topics, feel free to ask!

In [12]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "My name is Ed. What do you know about me?")
    display(Markdown(result.final_output))

I know that you, Ed, are an LLM engineer. You are teaching a course about AI Agents and you are knowledgeable about the MCP protocol. Is there anything else you'd like to share or ask about?

In [19]:
env = {"EXA_API_KEY": os.getenv("EXA_API_KEY")}
params = {"command": "npx", "args": ["-y", "mcp-remote", "https://mcp.exa.ai/mcp"], "env": env}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='web_search_exa', description='Search the web using Exa AI - performs real-time web searches and can scrape content from specific URLs. Supports configurable result counts and returns the content from the most relevant websites.', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Websearch query'}, 'numResults': {'type': 'number', 'description': 'Number of search results to return (default: 8)'}, 'livecrawl': {'type': 'string', 'enum': ['fallback', 'preferred'], 'description': "Live crawl mode - 'fallback': use live crawling as backup if cached content unavailable, 'preferred': prioritize live crawling (default: 'fallback')"}, 'type': {'type': 'string', 'enum': ['auto', 'fast', 'deep'], 'description': "Search type - 'auto': balanced search (default), 'fast': quick results, 'deep': comprehensive search"}, 'contextMaxCharacters': {'type': 'number', 'description': 'Maximum characters for context string optimized for LLMs (default: 10000)'

In [20]:
instructions = "You are able to search the web for information and briefly summarize the takeaways"
request = f"Please research the latest news on India Cements stock price and briefly summarize its outlook.\
           For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = "gpt-4.1-mini"

In [21]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

The latest news on India Cements stock price as of January 2026 indicates the following key points:

- India Cements experienced a significant share price decline of around 9% in January 2025 following weak Q3 financial results for FY25.
- The company reported a net loss widening substantially to Rs 429 crore in Q3 FY25, compared to a small loss in the same quarter the previous year.
- An exceptional loss of Rs 190 crore was also posted during this quarter.
- Revenue from operations dropped year-over-year by about 16.5% to Rs 903 crore in Q3 FY25.
- The weak financial performance triggered a steep drop in share price to an intraday low of Rs 315.80.

Summary Outlook:
India Cements is currently facing financial challenges with widened net losses and declining revenues impacting investor sentiment and share price negatively. The outlook appears cautious, as recent financial results have been weak. Investors may want to watch for signs of operational recovery or strategic changes going forward before considering positive stock price momentum.